# NeuroGolf 2026 Solver Development

This notebook turns EDA and diagnostics into the first input-derived solver candidates. It focuses on train-fit coverage before ONNX export, so each rule must explain every training pair for a task before it is considered for submission modeling.

# 1. Setup and Data Loading

## 1.1 Imports, Configuration, and Display Defaults

The notebook is self-contained for Kaggle. It recomputes lightweight task features and solver fits from attached competition JSON files rather than importing local project modules.

In [ ]:
from __future__ import annotations

import json
from collections import deque
from pathlib import Path
from typing import Any, Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

EXPECTED_TASK_COUNT = 400
BACKGROUND_COLOR = 0
MAX_DISPLAY_ROWS = 20
MAX_REVIEW_TASKS = 12
PLOT_CMAP = "viridis"
OUTPUT_DIR = (
    Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
)

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)


### 1.1 Configuration Notes

- `BACKGROUND_COLOR` controls the default background assumption for crop and component solvers.
- `MAX_REVIEW_TASKS` limits visual/manual review tables without changing solver coverage calculations.
- Solver outputs are written as lightweight CSVs so later ONNX notebooks can consume stable candidate lists.

## 1.2 Data Discovery and Normalization

The loader supports both one-file-per-task and combined JSON layouts. This keeps the notebook runnable when Kaggle input packaging changes but the task payload format stays the same.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
LOCAL_CANDIDATES = [
    Path("../input"),
    Path("input"),
    Path("data"),
    Path("../data"),
]


def candidate_roots() -> list[Path]:
    """Return input roots to scan for NeuroGolf JSON files.

    Returns:
        Candidate directories in priority order.
    """
    roots = [KAGGLE_INPUT] if KAGGLE_INPUT.exists() else []
    roots.extend(path for path in LOCAL_CANDIDATES if path.exists())
    return roots


def find_json_files() -> list[Path]:
    """Find task JSON files below candidate input roots.

    Returns:
        Sorted paths to discovered JSON files.
    """
    files: list[Path] = []
    for root in candidate_roots():
        files.extend(root.rglob("*.json"))
    return sorted(set(files))


def is_task_payload(obj: Any) -> bool:
    """Check whether a JSON object has the expected task layout.

    Args:
        obj: Parsed JSON object.

    Returns:
        True when the object contains train and test examples.
    """
    return isinstance(obj, dict) and "train" in obj and "test" in obj


def normalize_task_id(path: Path) -> str:
    """Build a stable task id from a JSON path.

    Args:
        path: Source JSON path.

    Returns:
        Normalized task id using the file stem.
    """
    return path.stem


def load_tasks(files: list[Path]) -> dict[str, dict[str, Any]]:
    """Load NeuroGolf tasks from one-file or combined JSON layouts.

    Args:
        files: JSON files discovered from the input roots.

    Returns:
        Mapping from task id to normalized task payload.
    """
    tasks: dict[str, dict[str, Any]] = {}
    for path in files:
        try:
            obj = json.loads(path.read_text())
        except Exception as exc:
            print(f"Skipping {path}: {exc}")
            continue

        if is_task_payload(obj):
            tasks[normalize_task_id(path)] = obj
        elif isinstance(obj, dict):
            for key, value in obj.items():
                if is_task_payload(value):
                    tasks[str(key)] = value
    return dict(sorted(tasks.items()))


json_files = find_json_files()
tasks = load_tasks(json_files)
print(f"Found {len(json_files):,} JSON files")
print(f"Loaded {len(tasks):,} tasks")


### 1.2 Loading Insights

- A healthy Kaggle run should load `400` normalized tasks from the competition input files.
- This notebook recomputes solver features locally so it can be run independently after EDA and baseline notebooks.
- If the task count is zero, attach the NeuroGolf competition dataset before interpreting solver coverage.

# 2. Solver Candidate Features

## 2.1 Task Structure and Object Features

This section creates the shared feature table used to decide which solver families should be attempted first.

In [ ]:
def arr(grid: list[list[int]]) -> np.ndarray:
    """Convert an ARC grid to an integer NumPy array.

    Args:
        grid: ARC-style integer grid.

    Returns:
        NumPy array with int64 dtype.
    """
    return np.asarray(grid, dtype=np.int64)


def grid_shape(grid: list[list[int]]) -> tuple[int, int]:
    """Return the row and column shape for a grid.

    Args:
        grid: ARC-style integer grid.

    Returns:
        Tuple of row count and column count.
    """
    x = arr(grid)
    return tuple(x.shape) if x.ndim == 2 else (0, 0)


def grid_colors(grid: list[list[int]]) -> set[int]:
    """Return the unique color tokens used in a grid.

    Args:
        grid: ARC-style integer grid.

    Returns:
        Set of integer color tokens.
    """
    x = arr(grid)
    return set(map(int, np.unique(x))) if x.size else set()


def connected_components(
    grid: np.ndarray, background: int = BACKGROUND_COLOR
) -> list[dict[str, Any]]:
    """Extract non-background connected components by color.

    Args:
        grid: Input grid array.
        background: Background color token.

    Returns:
        Component records with color, area, and bounding-box features.
    """
    visited = np.zeros(grid.shape, dtype=bool)
    components: list[dict[str, Any]] = []
    rows, cols = grid.shape
    for r in range(rows):
        for c in range(cols):
            if visited[r, c] or int(grid[r, c]) == background:
                continue
            color = int(grid[r, c])
            queue = deque([(r, c)])
            visited[r, c] = True
            coords = []
            while queue:
                cr, cc = queue.popleft()
                coords.append((cr, cc))
                neighbors = (
                    (cr - 1, cc),
                    (cr + 1, cc),
                    (cr, cc - 1),
                    (cr, cc + 1),
                )
                for nr, nc in neighbors:
                    if not (0 <= nr < rows and 0 <= nc < cols):
                        continue
                    if visited[nr, nc] or int(grid[nr, nc]) != color:
                        continue
                    visited[nr, nc] = True
                    queue.append((nr, nc))
            coord_arr = np.asarray(coords)
            r0, c0 = coord_arr.min(axis=0)
            r1, c1 = coord_arr.max(axis=0) + 1
            components.append(
                {
                    "color": color,
                    "area": len(coords),
                    "bbox": (int(r0), int(c0), int(r1), int(c1)),
                    "bbox_height": int(r1 - r0),
                    "bbox_width": int(c1 - c0),
                    "bbox_area": int((r1 - r0) * (c1 - c0)),
                }
            )
    return components


def task_features(task_id: str, task: dict[str, Any]) -> dict[str, Any]:
    """Build solver-development features for one task.

    Args:
        task_id: Stable task identifier.
        task: Normalized task payload.

    Returns:
        Dictionary of shape, palette, and component features.
    """
    train = task.get("train", [])
    test = task.get("test", [])
    input_shapes = [
        grid_shape(pair["input"]) for pair in train if "input" in pair
    ]
    output_shapes = [
        grid_shape(pair["output"]) for pair in train if "output" in pair
    ]
    input_colors = (
        set().union(*(grid_colors(pair["input"]) for pair in train))
        if train
        else set()
    )
    output_colors = (
        set().union(
            *(
                grid_colors(pair["output"])
                for pair in train
                if "output" in pair
            )
        )
        if train
        else set()
    )
    input_areas = [rows * cols for rows, cols in input_shapes]
    output_areas = [rows * cols for rows, cols in output_shapes]
    component_counts = [
        len(connected_components(arr(pair["input"])))
        for pair in train
        if "input" in pair
    ]
    shape_changes = [i != o for i, o in zip(input_shapes, output_shapes)]
    max_input_area = max(input_areas, default=0)
    max_output_area = max(output_areas, default=0)
    area_ratio = max_output_area / max_input_area if max_input_area else np.nan
    return {
        "task_id": task_id,
        "n_train": len(train),
        "n_test": len(test),
        "shape_changes_in_train": any(shape_changes),
        "max_input_area": max_input_area,
        "max_output_area": max_output_area,
        "area_ratio": area_ratio,
        "n_input_colors": len(input_colors),
        "n_output_colors": len(output_colors),
        "introduced_colors": tuple(sorted(output_colors - input_colors)),
        "removed_colors": tuple(sorted(input_colors - output_colors)),
        "max_components": max(component_counts, default=0),
        "mean_components": (
            float(np.mean(component_counts)) if component_counts else 0.0
        ),
    }


features_df = pd.DataFrame(
    [task_features(task_id, task) for task_id, task in tasks.items()]
)
if not features_df.empty:
    display(features_df.head(min(10, MAX_DISPLAY_ROWS)))
    display(
        features_df["shape_changes_in_train"]
        .value_counts()
        .rename_axis("shape_changes_in_train")
        .reset_index(name="task_count")
    )
else:
    display(features_df)


### 2.1 Feature Insights

- The latest EDA run showed `138 / 400` shape-changing tasks, so solver development must split same-shape and shape-changing rules early.
- Component counts are used here as routing signals: low-component tasks are candidates for object movement/selection, while high-component tasks likely need pattern or counting logic.
- Palette deltas help separate recoloring tasks from tasks that create or remove semantic tokens.

# 3. Same-Shape Solvers

## 3.1 Train-Fit Rule Checks

These solvers only become candidates when they fit every train pair in a task. They are intentionally strict so coverage numbers are trustworthy.

In [ ]:
def equal_grid(a: np.ndarray, b: np.ndarray) -> bool:
    """Compare two grids for equal shape and values.

    Args:
        a: First grid array.
        b: Second grid array.

    Returns:
        True when both arrays have identical shape and values.
    """
    return a.shape == b.shape and np.array_equal(a, b)


def infer_global_color_map(
    pairs: list[dict[str, Any]],
) -> dict[int, int] | None:
    """Infer a consistent input-to-output color mapping.

    Args:
        pairs: Training pairs to inspect.

    Returns:
        Color mapping when consistent, otherwise None.
    """
    mapping: dict[int, int] = {}
    for pair in pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if x.shape != y.shape:
            return None
        for src, dst in zip(x.ravel(), y.ravel()):
            src_i = int(src)
            dst_i = int(dst)
            if src_i in mapping and mapping[src_i] != dst_i:
                return None
            mapping[src_i] = dst_i
    return mapping


def apply_color_map(x: np.ndarray, mapping: dict[int, int]) -> np.ndarray:
    """Apply a color-token mapping to a grid.

    Args:
        x: Input grid array.
        mapping: Mapping from source token to destination token.

    Returns:
        Recolored grid array.
    """
    y = x.copy()
    for src, dst in mapping.items():
        y[x == src] = dst
    return y


def solver_global_color_map(train_pairs: list[dict[str, Any]]) -> bool:
    """Check whether one global color map fits all training pairs.

    Args:
        train_pairs: Training examples for one task.

    Returns:
        True when a consistent recoloring solves all pairs.
    """
    mapping = infer_global_color_map(train_pairs)
    if mapping is None:
        return False
    return all(
        equal_grid(
            apply_color_map(arr(pair["input"]), mapping), arr(pair["output"])
        )
        for pair in train_pairs
    )


def solver_background_to_single_color(
    train_pairs: list[dict[str, Any]], background: int = BACKGROUND_COLOR
) -> bool:
    """Check for a same-shape background-fill transformation.

    Args:
        train_pairs: Training examples for one task.
        background: Background color token.

    Returns:
        True when changed cells are background recolored to one token.
    """
    if not train_pairs:
        return False
    fill_color: int | None = None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if x.shape != y.shape:
            return False
        changed = x != y
        if not np.any(changed):
            continue
        if not np.all(x[changed] == background):
            return False
        output_colors = set(map(int, np.unique(y[changed])))
        if len(output_colors) != 1:
            return False
        current = next(iter(output_colors))
        if fill_color is None:
            fill_color = current
        elif fill_color != current:
            return False
    return fill_color is not None


def solver_geometric_transform(
    train_pairs: list[dict[str, Any]],
    transform: Callable[[np.ndarray], np.ndarray],
) -> bool:
    """Check whether a geometric transform fits all training pairs.

    Args:
        train_pairs: Training examples for one task.
        transform: Function that transforms an input grid array.

    Returns:
        True when the transform matches every output grid.
    """
    return bool(train_pairs) and all(
        equal_grid(transform(arr(pair["input"])), arr(pair["output"]))
        for pair in train_pairs
    )


same_shape_solvers: dict[str, Callable[[list[dict[str, Any]]], bool]] = {
    "background_to_single_color": solver_background_to_single_color,
    "global_color_map": solver_global_color_map,
    "flip_horizontal": lambda pairs: solver_geometric_transform(
        pairs, np.fliplr
    ),
    "flip_vertical": lambda pairs: solver_geometric_transform(
        pairs, np.flipud
    ),
    "rotate_90": lambda pairs: solver_geometric_transform(
        pairs, lambda x: np.rot90(x, 1)
    ),
    "rotate_180": lambda pairs: solver_geometric_transform(
        pairs, lambda x: np.rot90(x, 2)
    ),
    "rotate_270": lambda pairs: solver_geometric_transform(
        pairs, lambda x: np.rot90(x, 3)
    ),
    "transpose": lambda pairs: solver_geometric_transform(pairs, np.transpose),
}

same_shape_rows: list[dict[str, Any]] = []
for task_id, task in tasks.items():
    train_pairs = task.get("train", [])
    row = {"task_id": task_id}
    for solver_name, solver_fn in same_shape_solvers.items():
        row[solver_name] = bool(solver_fn(train_pairs))
    same_shape_rows.append(row)

same_shape_df = pd.DataFrame(same_shape_rows)
if not same_shape_df.empty:
    solver_cols = [col for col in same_shape_df.columns if col != "task_id"]
    same_shape_df["any_same_shape_solver"] = same_shape_df[solver_cols].any(
        axis=1
    )
    same_shape_coverage = (
        same_shape_df[solver_cols + ["any_same_shape_solver"]]
        .sum()
        .sort_values(ascending=False)
        .rename_axis("solver")
        .reset_index(name="task_count")
    )
    display(same_shape_coverage)
    display(
        same_shape_df.query("any_same_shape_solver").head(MAX_DISPLAY_ROWS)
    )
else:
    display(same_shape_df)


### 3.1 Same-Shape Solver Insights

- The latest diagnostics run found `62` tasks compatible with at least one strict same-shape solver.
- `background_to_single_color` is the first implementation target because it covers `50` train-fit tasks.
- Global color maps and simple geometric transforms are useful but narrow; they should be implemented as small, auditable solver families rather than treated as the main strategy.

# 4. Shape-Changing Solvers

## 4.1 Crop, Scale, and Tile Checks

These checks target the strongest shape-changing motifs from EDA: compression, expansion, and construction.

In [ ]:
def crop_to_non_background(
    x: np.ndarray, background: int = BACKGROUND_COLOR
) -> np.ndarray | None:
    """Crop a grid to the non-background bounding box.

    Args:
        x: Input grid array.
        background: Background color token.

    Returns:
        Cropped array, or None when no non-background cells exist.
    """
    coords = np.argwhere(x != background)
    if coords.size == 0:
        return None
    r0, c0 = coords.min(axis=0)
    r1, c1 = coords.max(axis=0) + 1
    return x[r0:r1, c0:c1]


def solver_crop_non_background(train_pairs: list[dict[str, Any]]) -> bool:
    """Check whether non-background cropping fits all training pairs.

    Args:
        train_pairs: Training examples for one task.

    Returns:
        True when cropping each input matches the output.
    """
    if not train_pairs:
        return False
    for pair in train_pairs:
        cropped = crop_to_non_background(arr(pair["input"]))
        if cropped is None or not equal_grid(cropped, arr(pair["output"])):
            return False
    return True


def integer_scale_factor(
    input_shape: tuple[int, int], output_shape: tuple[int, int]
) -> tuple[int, int] | None:
    """Infer an integer row and column scale factor.

    Args:
        input_shape: Input grid shape.
        output_shape: Output grid shape.

    Returns:
        Row and column scale factors, or None if not integral.
    """
    in_rows, in_cols = input_shape
    out_rows, out_cols = output_shape
    if in_rows == 0 or in_cols == 0:
        return None
    if out_rows % in_rows != 0 or out_cols % in_cols != 0:
        return None
    return out_rows // in_rows, out_cols // in_cols


def solver_nearest_integer_scale(train_pairs: list[dict[str, Any]]) -> bool:
    """Check whether nearest-neighbor integer scaling fits a task.

    Args:
        train_pairs: Training examples for one task.

    Returns:
        True when one integer scale factor fits every pair.
    """
    if not train_pairs:
        return False
    scale: tuple[int, int] | None = None
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        current = integer_scale_factor(tuple(x.shape), tuple(y.shape))
        if current is None or current == (1, 1):
            return False
        if scale is None:
            scale = current
        elif scale != current:
            return False
        scaled = np.repeat(
            np.repeat(x, current[0], axis=0), current[1], axis=1
        )
        if not equal_grid(scaled, y):
            return False
    return True


def solver_periodic_tile_from_input(train_pairs: list[dict[str, Any]]) -> bool:
    """Check whether tiled input repetition fits all outputs.

    Args:
        train_pairs: Training examples for one task.

    Returns:
        True when tiling each input recreates its output.
    """
    if not train_pairs:
        return False
    for pair in train_pairs:
        x = arr(pair["input"])
        y = arr(pair["output"])
        if y.shape[0] < x.shape[0] or y.shape[1] < x.shape[1] or x.size == 0:
            return False
        repeats = (
            int(np.ceil(y.shape[0] / x.shape[0])),
            int(np.ceil(y.shape[1] / x.shape[1])),
        )
        tiled = np.tile(x, repeats)[: y.shape[0], : y.shape[1]]
        if not equal_grid(tiled, y):
            return False
    return True


shape_solvers: dict[str, Callable[[list[dict[str, Any]]], bool]] = {
    "crop_non_background": solver_crop_non_background,
    "nearest_integer_scale": solver_nearest_integer_scale,
    "periodic_tile_from_input": solver_periodic_tile_from_input,
}

shape_rows: list[dict[str, Any]] = []
for task_id, task in tasks.items():
    train_pairs = task.get("train", [])
    row = {"task_id": task_id}
    for solver_name, solver_fn in shape_solvers.items():
        row[solver_name] = bool(solver_fn(train_pairs))
    shape_rows.append(row)

shape_solver_df = pd.DataFrame(shape_rows)
if not shape_solver_df.empty:
    shape_cols = [col for col in shape_solver_df.columns if col != "task_id"]
    shape_solver_df["any_shape_solver"] = shape_solver_df[shape_cols].any(
        axis=1
    )
    shape_coverage = (
        shape_solver_df[shape_cols + ["any_shape_solver"]]
        .sum()
        .sort_values(ascending=False)
        .rename_axis("solver")
        .reset_index(name="task_count")
    )
    display(shape_coverage)
    display(shape_solver_df.query("any_shape_solver").head(MAX_DISPLAY_ROWS))
else:
    display(shape_solver_df)


### 4.1 Shape-Changing Solver Insights

- The latest diagnostics run found only `4` strict shape-changing train-fit tasks from crop, integer scale, and tile checks.
- That low count is useful: naive crop/scale/tile is too weak for the `138` shape-changing tasks and must be expanded with object selection and construction logic.
- The next refinement should classify strong compression into object crop, selected-object output, count summary, and fixed-template output.

# 5. Candidate Table and Next Actions

## 5.1 Build the Solver Candidate Table

The candidate table joins features and strict solver fits, then assigns the next recommended action for each task.

In [ ]:
def assign_next_action(row: pd.Series) -> str:
    """Assign a next modeling action from task features and solver fits.

    Args:
        row: Joined solver candidate row.

    Returns:
        Recommended next action label.
    """
    if bool(row.get("any_same_shape_solver", False)):
        return "export simple same-shape solver"
    if bool(row.get("any_shape_solver", False)):
        return "export simple shape solver"
    if (
        bool(row.get("shape_changes_in_train", False))
        and row.get("area_ratio", 1.0) < 0.75
    ):
        return "deep dive crop/extract/compress"
    if (
        bool(row.get("shape_changes_in_train", False))
        and row.get("area_ratio", 1.0) > 1.25
    ):
        return "deep dive expand/tile/construct"
    if row.get("max_components", 999) <= 10:
        return "deep dive object movement/selection"
    return "deep dive pattern/counting/global logic"


candidate_df = features_df.copy()
if not candidate_df.empty:
    candidate_df = candidate_df.merge(same_shape_df, on="task_id", how="left")
    candidate_df = candidate_df.merge(
        shape_solver_df, on="task_id", how="left"
    )
    bool_cols = [
        col
        for col in candidate_df.columns
        if col.startswith(("background_", "global_", "flip_", "rotate_"))
        or col in {"transpose", "any_same_shape_solver", "any_shape_solver"}
        or col in shape_solvers
    ]
    candidate_df[bool_cols] = (
        candidate_df[bool_cols].fillna(False).astype(bool)
    )
    candidate_df["next_action"] = candidate_df.apply(
        assign_next_action, axis=1
    )
    action_table = (
        candidate_df["next_action"]
        .value_counts()
        .rename_axis("next_action")
        .reset_index(name="task_count")
    )
    display(action_table)
    display(candidate_df.head(MAX_DISPLAY_ROWS))
else:
    candidate_df = pd.DataFrame()
    action_table = pd.DataFrame()
    display(candidate_df)


### 5.1 Candidate Table Insights

- Candidate routing should make the next notebooks narrower: export simple train-fit rules first, then deep dive the larger unsolved buckets.
- The latest diagnostics indicate the biggest remaining work is object movement/selection and crop/extract/compress, not more generic baseline packaging.
- Every future ONNX export should be traceable back to this table through `task_id`, `next_action`, and the fitted solver flags.

## 5.2 Export Solver Candidate Artifacts

The exported CSVs are lightweight handoff artifacts for ONNX export and failure-analysis notebooks.

In [ ]:
if not candidate_df.empty:
    candidate_path = OUTPUT_DIR / "neurogolf_solver_candidate_table.csv"
    same_shape_path = OUTPUT_DIR / "neurogolf_same_shape_solver_fits.csv"
    shape_path = OUTPUT_DIR / "neurogolf_shape_solver_fits.csv"
    candidate_df.to_csv(candidate_path, index=False)
    same_shape_df.to_csv(same_shape_path, index=False)
    shape_solver_df.to_csv(shape_path, index=False)
    print(f"Wrote {candidate_path}")
    print(f"Wrote {same_shape_path}")
    print(f"Wrote {shape_path}")
else:
    print(
        "No solver candidate artifacts were written because no tasks were loaded."
    )


### 5.2 Export Insights

- `neurogolf_solver_candidate_table.csv` is the main artifact for deciding which tasks move into simple ONNX export versus deeper analysis.
- The solver-fit CSVs preserve exact task ids per rule family, which keeps coverage claims auditable.
- After this notebook runs, the next step is an ONNX export notebook for the simple same-shape solvers that already fit train pairs.